In [14]:
"""Lunar Lander : train a Lunar Lander, to land correctly on the moon."""
from pprint import pprint

import gymnasium as gym

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login 

from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# How Gymnasium Works

- We create our environment using `gym.make()`
- We reset the environment to its initial state with `env.reset()`

Then, at each step
- Get an action using our model
- Using `env.step(action)`, we perform this action in the environment and get:
    - `observation` : The new state (st+1)
    - `reward` : The reward we get after executing the action
    - `terminated` : Indicates if the episode terminated (agent reach the terminal state)
    - `truncated` : Indicates a time limit or if an agent go out of bounds of the environment
    - `info` : A dictionary that provides additional information (depends on the environment)

In [28]:
# First, we create our environment called LunarLander-v2
env = gym.make("LunarLander-v2")

# Then we reset this environment
observation, info = env.reset()

# see what the Environment looks like
# The observation is a vector of size 8
# each value contains different information about the lander
print("Observation Space Shape", env.observation_space.shape)

# Get a random observation
obs = env.observation_space.sample()
obs_fields = [
    "x",
    "y", 
    "x_velocity",
    "y_velocity",
    "angle",
    "angular_velocity",
    "left_leg_contact",
    "right_leg_contact"
]
print("Sample observation: ")
pprint(
    dict(zip(obs_fields, env.observation_space.sample(), strict=True)),
    sort_dicts=False
)

# The action space is a discrete space of 4 possible actions
print("\nAction Space Shape", env.action_space.n)
actions = [
    "Do nothing",
    "Fire left orientation engine",
    "Fire main engine",
    "Fire right orientation engine"
]
for _ in range(20):
    # Get a random action
    action = env.action_space.sample()

    # Perform the action
    print("Action taken:", actions[action])
    observation, reward, terminated, truncated, info = env.step(action)

    # If the game is terminated (in our case we land, crashed) or truncated (timeout)
    if terminated or truncated:
        # Reset the environment
        print("Environment is reset")
        observation, info = env.reset()

env.close()

Observation Space Shape (8,)
Sample observation: 
{'x': 47.587025,
 'y': -4.108406,
 'x_velocity': 1.0766854,
 'y_velocity': 3.0359592,
 'angle': -0.01860175,
 'angular_velocity': 2.9174118,
 'left_leg_contact': 0.022134554,
 'right_leg_contact': 0.20811054}

Action Space Shape 4
Action taken: Fire right orientation engine
Action taken: Fire right orientation engine
Action taken: Fire left orientation engine
Action taken: Fire right orientation engine
Action taken: Fire left orientation engine
Action taken: Fire main engine
Action taken: Fire main engine
Action taken: Fire left orientation engine
Action taken: Fire main engine
Action taken: Fire main engine
Action taken: Fire main engine
Action taken: Fire right orientation engine
Action taken: Fire main engine
Action taken: Do nothing
Action taken: Fire right orientation engine
Action taken: Fire right orientation engine
Action taken: Do nothing
Action taken: Fire main engine
Action taken: Do nothing
Action taken: Fire right orientati

## Vectorized Environment
We create a vectorized environment (stack multiple independent environments into a single environment), this way, we’ll have more diverse experiences during the training.

In [27]:
vec_env = make_vec_env("LunarLander-v2", n_envs=16)
vec_env.num_envs

16